# Smoke inference: one migration through the pipeline on a Colab T4

What this does: clone the repo, load **Qwen2.5-Coder-7B-Instruct** (base model, no adapter) in 4-bit at the
revision pinned in `pipeline/model.lock.json`, translate `harness/tests/smoke_io` from Python to Haskell with the
frozen prompt `v1`, and show the output plus timing. Use the timing to size the full matrix run.

Before running: **Runtime → Change runtime type → T4 GPU.** No Hugging Face token is needed for this notebook
(the base model is public); the `HF_TOKEN` secret is only read if present, for later adapter pushes.

Everything this notebook needs (pipeline, prompt, harness smoke tests) is on `main`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
REPO = "https://github.com/05ryt31/CS152-Project.git"
BRANCH = "main"

if not os.path.isdir("CS152-Project"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO], check=True)
os.chdir("CS152-Project")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
print(sorted(os.listdir("harness/tests/smoke_io")))

In [ ]:
%pip install -q "transformers>=4.45,<5" accelerate peft bitsandbytes
# transformers is pinned below 5: 5.x changed apply_chat_template's return type (Ellen hit an AttributeError on 5.20.1; 4.49 works)

In [ ]:
# Optional: Hugging Face login from the Colab secret (only needed later, to push the adapter).
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    print("no HF_TOKEN secret; continuing without login (base model is public)")

In [ ]:
# Pipeline self-check (no model involved): template parsing, extraction rule, dry end-to-end run.
!python pipeline/test_pipeline.py

In [ ]:
# The real run: Python -> Haskell on smoke_io with the 4-bit base model. First call also downloads ~5.5 GB.
!python pipeline/migrate.py --run-id smoke01 --problems-dir harness/tests --problem smoke_io \
    --pairs python:haskell --backend hf --load-4bit

In [ ]:
import json, pathlib
out = pathlib.Path("results/raw/smoke01/smoke_io/python_to_haskell")
meta = json.loads((out / "meta.json").read_text())
print({k: meta[k] for k in ["status", "model", "extraction", "duration_s", "usage", "prompt_version"]})
print("\n--- generated", meta.get("target_file"))
print((out / meta["target_file"]).read_text())
print("--- raw reply (first 1500 chars)")
print(meta["raw_response"][:1500])

## Sizing the matrix

`duration_s` above includes model loading only on the first call inside a process; the pipeline keeps the model loaded
across pairs, so the per-translation time is what matters. Multiply by the counts in `pipeline/MATRIX.md`
(15-20 problems → 330-440 priority-1 translations) and compare with the 8 GPU-hour bound in `pipeline/MODEL.md`.

In [ ]:
# Optional: time several pairs in one process to get a steadier per-translation estimate.
!python pipeline/migrate.py --run-id smoke02 --problems-dir harness/tests --problem smoke_io \
    --pairs python:prolog,python:java,java:haskell,haskell:python --backend hf --load-4bit
import glob, json, statistics
d = [json.load(open(p))["duration_s"] for p in glob.glob("results/raw/smoke02/smoke_io/*/meta.json")]
print("per-translation seconds:", d, "median", statistics.median(d))
for n in (330, 440, 600):
    print(f"{n} translations ~ {n * statistics.median(d) / 3600:.1f} GPU-hours")

## Checking the output with the harness

The harness runs Docker, which Colab does not provide. Copy a pair folder back to a machine with Docker and run it from
`harness/`:

```
./run_harness.sh -f ../results/raw/smoke01/smoke_io/python_to_haskell -p -a -v -s -l
```

(`-p -a -v -s -l` skip Python, JS, Java, Scheme, Prolog so only Haskell runs.) The cell below zips the results for download
or copies them to Drive.

In [ ]:
!zip -qr results_smoke.zip results/raw && ls -la results_smoke.zip
# Or keep them on Drive:
# from google.colab import drive; drive.mount("/content/drive")
# !cp results_smoke.zip /content/drive/MyDrive/CS152/